# 🕵️ Dia 74 — Entrevistando vítimas de fraude sem revitimizá-las

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 74* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-74.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quinta, 29/out, agência Centro. Seu Antônio, 71 anos, chega com o extrato impresso e um papel dobrado com o telefone de um "Dr. Silva — recupera Pix". Antes de sentar: *"Eu sei que fui burro. Pode falar."*

A primeira pergunta do seu roteiro era "por que o senhor não desconfiou?". Rafa balança a cabeça, devagar.

> ⚠️ O Seu Antônio é **fictício**. Neste dia, quem o interpreta é um colega, num treino. Não use a experiência real de ninguém.

Neste notebook você vai:

1. (Opcional) **Transcrever** sua entrevista de treino com Whisper.
2. **Classificar as perguntas** de uma transcrição com um classificador simples em pt-BR, e corrigir o que ele errar.
3. **Contar os tipos** e conferir o Checkpoint.
4. **Jogar o quiz da reescrita**.
5. **Gerar o plano de entrevista** do Seu Antônio.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo aqui é fictício (Operação Isca).

## 0. (Opcional) Transcrever com Whisper

Só se você gravou a entrevista de treino **com consentimento**. Suba o áudio no Colab (ícone de pasta) e rode a célula abaixo; as linhas com `!` instalam e rodam o Whisper. Sem isso, o notebook segue com a transcrição de exemplo. **Apague o áudio ao terminar a auditoria e avise o seu colega.**

In [ ]:
# CÉLULA OPCIONAL — precisa de rede e de alguns minutos. Pode pular.
# !pip install -q -U openai-whisper
# !apt-get -qq install -y ffmpeg
# !whisper entrevista-treino.m4a --model small --language pt --output_format json
print("Se rodou o Whisper, o arquivo entrevista-treino.json está na pasta. Senão, seguimos com o exemplo.")

## 1. A transcrição

Usamos, nesta ordem: o JSON do seu Whisper (se existir), o CSV do repositório, ou a transcrição de exemplo embutida. A de exemplo tem **erros colocados de propósito**: é para auditar, não para imitar.

Como o Whisper não separa quem fala, no seu JSON você marca à mão quem é "E" (entrevistador) e quem é "A" (vítima). Aqui, para simplificar, tratamos como pergunta do entrevistador toda fala sua que termina em `?`.

In [ ]:
import csv, io, json, urllib.request
from pathlib import Path

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-74/transcricao-roleplay.csv")

# Plano B offline: (falante, fala). E = entrevistador; A = Seu Antônio (interpretado).
TRANSCRICAO_EMBUTIDA = [
    ('E', 'Bom dia, seu Antônio. Meu nome é [analista], trabalho na Vigia, a empresa que a Serra Azul contratou para entender esse golpe. A Marta, da cooperativa, combinou esta conversa com o senhor.'),
    ('E', 'Antes de começar: o que o senhor contar vai para o relatório do caso, que a cooperativa usa para tentar recuperar o dinheiro e para proteger outros cooperados. O senhor pode parar ou fazer uma pausa quando quiser.'),
    ('E', 'E uma coisa importante: nem eu nem ninguém da cooperativa vai pedir dinheiro, taxa, senha ou código ao senhor. Se alguém pedir, é golpe. Tudo bem começarmos?'),
    ('A', 'Tudo bem. Eu tô com vergonha, viu. Minha filha nem sabe ainda.'),
    ('E', 'Por onde o senhor gostaria de começar?'),
    ('A', 'Começou com um áudio do Cláudio, no domingo à noite. Era a voz dele, eu conheço o Cláudio há doze anos.'),
    ('E', 'Me conte como foi esse áudio.'),
    ('A', 'Ele disse que ia me ligar cedo por causa do Pix bloqueado. E ligou. Falou que tinha gente tentando tirar dinheiro da minha conta e que eu tinha que passar pra uma conta de segurança.'),
    ('E', 'E o que aconteceu depois disso?'),
    ('A', 'Ele foi me guiando no aplicativo. Eu fiz um Pix, aí ele disse que não tinha ido tudo e eu fiz outro.'),
    ('E', 'O senhor não achou estranho transferir para uma conta que não era sua?'),
    ('A', 'Achei... mas era o Cláudio, né. Ele falou que não podia desligar senão bloqueava tudo.'),
    ('E', 'Que nome aparecia como recebedor quando o senhor fez o Pix?'),
    ('A', "Era um nome de pessoa, não lembro. Ele disse que era o nome do 'responsável pela segurança'."),
    ('E', 'E ele pediu para o senhor não contar isso pra ninguém, não pediu?'),
    ('A', 'Pediu... acho que pediu, sim.'),
    ('E', 'Quando foi, quanto foi e para qual chave?'),
    ('A', 'Ah, não sei... foi de manhã. Quase dez mil, nos dois.'),
    ('E', 'Tudo bem não lembrar tudo. Podemos olhar juntos o extrato depois, um Pix de cada vez. O senhor tem o histórico de chamadas no celular?'),
    ('A', 'Tenho, ficou lá.'),
    ('E', 'Alguém ainda está entrando em contato com o senhor sobre isso? Alguém pediu mais dinheiro?'),
    ('A', 'Ontem ligou um doutor, advogado, dizendo que recupera Pix. Queria oitocentos reais adiantado.'),
    ('E', 'Obrigado por me contar. Esse tipo de ligação é comum depois de um golpe, e é outro golpe. Ele mandou nome, número ou site?'),
    ('A', 'Mandou um número. Tá anotado num papel.'),
    ('E', "Deixa eu resumir o que entendi, e o senhor me corrige: áudio no domingo à noite, ligação na segunda cedo, dois Pix guiados por telefone, e ontem a ligação do 'advogado'. Está certo?"),
    ('A', 'Tá certo.'),
    ('E', 'Os próximos passos: a cooperativa já abriu o pedido de devolução pelo Pix, eu levo esses dados para o relatório, e a Marta liga para o senhor pelo número do seu cadastro. Não posso prometer que o dinheiro volta; quem decide isso é o processo entre as instituições. Como o senhor está se sentindo agora?'),
]

if Path("entrevista-treino.json").is_file():                       # 1º: o seu Whisper
    segs = json.loads(Path("entrevista-treino.json").read_text(encoding="utf-8"))["segments"]
    FALAS = [("E" if s["text"].strip().endswith("?") else "A", s["text"].strip()) for s in segs]
    print("✅ Usando a SUA transcrição (marque à mão quem fala, se precisar).")
else:
    try:                                                           # 2º: o CSV do repositório
        with urllib.request.urlopen(URL, timeout=10) as r:
            FALAS = [(l["falante"], l["fala"]) for l in csv.DictReader(io.StringIO(r.read().decode("utf-8")))]
        print("✅ Transcrição de exemplo baixada do repositório.")
    except Exception as erro:                                      # 3º: a embutida
        FALAS = TRANSCRICAO_EMBUTIDA
        print(f"⚠️ Sem acesso ({type(erro).__name__}); usando a transcrição embutida.")

for i, (quem, fala) in enumerate(FALAS, start=1):
    print(f"{i:02d} {quem}: {fala}")

## 2. Um classificador de perguntas (ajuda, não juiz)

As regras vêm da tabela de tipos de pergunta do original, adaptadas ao português. A **ordem** importa: primeiro procuramos os tipos a evitar (culpabilizante, indutora, múltipla), porque uma pergunta como "por que você não...?" também é "aberta" na forma, mas fecha a pessoa.

Ele vai errar. A auditoria é sua: confira cada linha.

In [ ]:
import re

REGRAS = [
    # (tipo, padrão, por que é desse tipo)
    ("culpabilizante", r"^por que (o senhor |a senhora |você )?(não|nao)|não achou estranho|não desconfiou|deveria ter|você realmente|acreditou mesmo",
     "julga a decisão da vítima"),
    ("indutora", r", (não (pediu|foi|é|disse)|certo|né|não é mesmo)\?$",
     "sugere a resposta e planta o detalhe"),
    ("múltipla", r"\b(quando|quanto|qual|quem|onde)\b.*,.*\b(quando|quanto|qual|quem|onde|para qual)\b",
     "várias perguntas numa só"),
    ("convite aberto", r"por onde .*começar|me cont[ae]|pode me contar|gostaria de contar",
     "deixa a pessoa escolher o caminho"),
    ("estímulo aberto", r"o que aconteceu depois|e depois|o que mais|como foi",
     "estende o relato"),
    ("específica", r"^(que|qual|quais|como|quem|onde|quando|quanto|alguém)\b",
     "preenche lacuna sem sugerir resposta"),
    ("fechada", r"^(o senhor|a senhora|você|ele|ela|foi|tem|tinha|era|está|tudo bem)\b",
     "sim/não: só para confirmar o que já foi dito"),
]

def classificar(pergunta):
    """Devolve (tipo, motivo) da primeira regra que casar; senão, 'revisar à mão'."""
    p = pergunta.strip().lower()
    for tipo, padrao, motivo in REGRAS:
        if re.search(padrao, p):
            return tipo, motivo
    return "revisar à mão", "nenhuma regra casou"

def perguntas_de(falas):
    """Extrai do entrevistador as perguntas e os pedidos de relato (uma fala pode ter vários)."""
    saida = []
    for n, (quem, fala) in enumerate(falas, start=1):
        if quem != "E":
            continue
        for frase in re.findall(r"[^.?!]*[.?!]", fala):        # cada frase da fala
            frase = frase.strip()
            # pergunta = termina em '?' OU é um pedido de relato ("Me conte...", "Pode me contar...")
            if frase.endswith("?") or re.match(r"(me conte|pode me contar)", frase.lower()):
                saida.append((n, frase))
    return saida

PERGUNTAS = [(n, q, *classificar(q)) for n, q in perguntas_de(FALAS)]
for n, q, tipo, motivo in PERGUNTAS:
    marca = "🟥" if tipo in ("culpabilizante", "indutora", "múltipla") else "🟩"
    print(f"{marca} fala {n:02d} [{tipo:15s}] {q}")

### ✍️ Corrija o classificador

Se discordar de alguma linha, escreva a correção no dicionário abaixo (número da fala → tipo) e rode. É assim que se faz uma auditoria: a máquina propõe, você decide e registra.

In [ ]:
CORRECOES = {
    # 17: "específica",      # exemplo: descomente e ajuste se discordar do classificador
}

TIPOS_FINAIS = []
for n, q, tipo, motivo in PERGUNTAS:
    tipo_final = CORRECOES.get(n, tipo)
    TIPOS_FINAIS.append((n, q, tipo_final))
print(f"{len(TIPOS_FINAIS)} perguntas auditadas; {len(CORRECOES)} correções manuais registradas.")

## 3. A tabela de contagem e o Checkpoint

O Checkpoint do original, em código: pelo menos metade das perguntas **antes do Encerramento** são abertas; zero indutoras; zero culpabilizantes. Consideramos Encerramento a fala que começa com "Deixa eu resumir".

In [ ]:
from collections import Counter

inicio_encerramento = next((n for n, (q, f) in enumerate(FALAS, start=1)
                            if q == "E" and f.lower().startswith("deixa eu resumir")), len(FALAS) + 1)
antes = [t for n, q, t in TIPOS_FINAIS if n < inicio_encerramento]
contagem = Counter(t for _, _, t in TIPOS_FINAIS)

print("Tipo               Quantidade")
for tipo, qtd in contagem.most_common():
    print(f"{tipo:18s} {qtd}")

abertas = sum(t in ("convite aberto", "estímulo aberto") for t in antes)
print(f"\nAbertas antes do Encerramento: {abertas}/{len(antes)} "
      f"{'✅' if antes and abertas >= len(antes) / 2 else '❌ (meta: pelo menos metade)'}")
for ruim in ("indutora", "culpabilizante"):
    print(f"{ruim.capitalize():15s}: {contagem.get(ruim, 0)} {'✅' if not contagem.get(ruim) else '❌ reescreva cada uma'}")

print("\nPerguntas para reescrever:")
for n, q, t in TIPOS_FINAIS:
    if t in ("indutora", "culpabilizante", "múltipla"):
        print(f"  fala {n:02d} [{t}] {q}")

> 🧠 **Repare na fala logo depois da indutora.** "Pediu... acho que pediu, sim." É o que o original avisa: a pergunta indutora planta o detalhe, e a resposta não vale nada como evidência. Se o sigilo importa para o caso (importa: é um mecanismo de golpe), a pergunta certa é aberta: *"O que ele disse sobre contar ou não contar para alguém?"*

A transcrição de exemplo também **não** chega à metade de perguntas abertas: depois do primeiro escorregão, o entrevistador desceu a escada e não subiu de volta. Ache o ponto em que isso aconteceu.

E repare onde estão os detalhes mais úteis da transcrição: o papel com o número do "Dr. Silva" veio depois de *"Alguém ainda está entrando em contato com o senhor?"* (item 6 do checklist de linguagem).

## 4. 🎮 Quiz da reescrita

Para cada pergunta ruim, escolha a melhor versão. As cinco primeiras vêm da Parte 1 do original (🇺🇸, traduzidas); as cinco últimas, do caso do Seu Antônio (🇧🇷). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("🇺🇸 'Por que você mandou dinheiro para alguém que nunca tinha visto?'",
     {"a": "Você não achou arriscado mandar dinheiro?", "b": "Pode me contar sobre a primeira vez em que o assunto dinheiro apareceu?", "c": "Quem te convenceu a mandar?"}, "b"),
    ("🇺🇸 'Ele disse que o retorno era garantido, certo?'",
     {"a": "O que ele disse sobre o retorno?", "b": "Ele garantiu o retorno ou não?", "c": "O retorno era garantido, não era?"}, "a"),
    ("🇺🇸 'Quanto você perdeu no total?'",
     {"a": "Foi mais de dez mil?", "b": "Quanto foi, em quantas vezes e para quem?", "c": "Podemos passar pelos pagamentos um de cada vez? Qualquer registro ajuda."}, "c"),
    ("🇺🇸 'A que horas exatamente foi a primeira ligação?'",
     {"a": "O que você lembra do momento da primeira ligação? O histórico do celular pode ajudar.", "b": "Foi de manhã ou de tarde? Precisa ser exato.", "c": "Por que não anotou o horário?"}, "a"),
    ("🇺🇸 'Você contou para o seu marido?'",
     {"a": "Seu marido sabe?", "b": "Tem alguém com quem você gostaria de contar nesse momento?", "c": "Não acha que deveria contar ao seu marido?"}, "b"),
    ("🇧🇷 'Por que o senhor não desconfiou da conta de segurança?'",
     {"a": "O que o Cláudio, ou quem usava a voz dele, disse sobre essa conta?", "b": "O senhor sabia que conta de segurança não existe?", "c": "O senhor não achou estranho?"}, "a"),
    ("🇧🇷 'Ele pediu para o senhor não contar para ninguém, não pediu?'",
     {"a": "Ele pediu segredo?", "b": "Ele pediu para não contar, certo?", "c": "O que ele disse sobre contar ou não contar para alguém?"}, "c"),
    ("🇧🇷 'Quando foi, quanto foi e para qual chave?'",
     {"a": "Vamos olhar juntos o extrato, um Pix de cada vez?", "b": "Me diz rápido a data, o valor e a chave.", "c": "Foi tudo na segunda-feira?"}, "a"),
    ("🇧🇷 (ele diz 'Eu sei que fui burro.') Você responde:",
     {"a": "Que nada, acontece com todo mundo, mas o senhor devia ter ligado na agência.", "b": "Esse golpe foi feito para enganar gente cuidadosa. Usaram a voz de alguém que o senhor conhece. Por onde o senhor gostaria de começar?", "c": "Vamos direto ao que interessa."}, "b"),
    ("🇧🇷 Sobre o 'Dr. Silva' que recupera Pix por R$ 800:",
     {"a": "Pague só metade adiantado.", "b": "Vou investigar se ele é advogado de verdade.", "c": "Ninguém da cooperativa nem da Vigia pede dinheiro. Recuperação mediante taxa adiantada é outro golpe. Não pague e não retorne a ligação."}, "c"),
]

acertos = 0
for i, (pergunta, opcoes, certa) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {pergunta}")
    for letra, txt in opcoes.items():
        print(f"   {letra}) {txt}")
    if MODO_JOGO:
        r = ""
        for _ in range(5):                                  # limite de tentativas: nunca trava
            r = input("   Melhor versão (a/b/c): ").strip().lower()
            if r in opcoes:
                break
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ era {certa}")
    else:
        print(f"   → {certa}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 5. 📝 O plano de entrevista do Seu Antônio

Fase **Planejamento e preparação** do PEACE, em uma página. Preencha e rode.

In [ ]:
#@title Plano de entrevista (PEACE) { display-mode: "form" }
local_e_horario = "Agência Centro, sala pequena, 14h, sem limite de tempo"  #@param {type:"string"}
acompanhante = "Perguntar se ele quer alguém de confiança junto; a escolha é dele"  #@param {type:"string"}
precisa_obter = "Cada Pix (data, valor, conta recebedora pelo comprovante); número do áudio e da ligação; se há contato em andamento"  #@param {type:"string"}
abertura = "Quem sou, por que estou aqui, o que acontece com o que ele contar, que pode pausar ou parar, e que ninguém da Vigia ou da cooperativa pede dinheiro, senha ou código"  #@param {type:"string"}
tratar_primeiro = "O 'Dr. Silva': golpe de recuperação em andamento; não pagar, não retornar a ligação"  #@param {type:"string"}
nao_perguntar_agora = "Família, saldo restante, detalhes do empréstimo (se houver); horário exato"  #@param {type:"string"}
proximos_passos = "Pedido de devolução já aberto pela cooperativa; quem decide é o processo entre as instituições; a Marta liga pelo número do cadastro"  #@param {type:"string"}
deixar_por_escrito = "Número oficial do verso do cartão + 'a Serra Azul nunca pede transferência, senha ou código'"  #@param {type:"string"}

import hashlib
from pathlib import Path

plano = f"""# Plano de entrevista — cooperado "Seu Antônio" (FICTÍCIO, treino)

## P — Planejamento e preparação
- Local e horário: {local_e_horario}
- Acompanhante: {acompanhante}
- O que preciso obter: {precisa_obter}
- O que NÃO pergunto agora: {nao_perguntar_agora}

## E — Engajar e explicar
- {abertura}
- Tratar primeiro: {tratar_primeiro}

## A — Relato
- Convite aberto: "Por onde o senhor gostaria de começar?"
- Estímulos: "E o que aconteceu depois?" / "Me conte mais sobre essa ligação."
- Específicas (só depois do relato livre): "Que nome aparecia como recebedor?" / "Vamos olhar o extrato, um Pix de cada vez?"
- Contato em andamento: "Alguém ainda está falando com o senhor sobre isso? Alguém pediu mais dinheiro?"

## C — Encerramento
- Resumir de volta e pedir correção.
- Próximos passos: {proximos_passos}
- Por escrito: {deixar_por_escrito}
- "Como o senhor está se sentindo agora?"

## E — Avaliação
- Auditoria de perguntas (meta: metade abertas, zero indutoras, zero culpabilizantes).
- Pergunta ao colega que interpretou: "Houve algum momento em que você se sentiu julgado?"
- Áudio apagado após a auditoria, e colega avisado.
"""

Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia-74-plano-entrevista.md")
saida.write_text(plano, encoding="utf-8")
print(plano)
print("💾", saida, "| SHA-256:", hashlib.sha256(saida.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: ficou em notas/)")

## ✅ Antes de fechar o Dia 74

- [ ] 15 perguntas reescritas (Parte 1)
- [ ] Entrevista de treino gravada com consentimento, transcrita, auditada; áudio apagado
- [ ] Contagem por tipo: metade aberta, zero indutora, zero culpabilizante
- [ ] Resposta do colega a "Houve algum momento em que você se sentiu julgado?" registrada como foi dada
- [ ] `notas/dia-74-plano-entrevista.md` gerado

**Amanhã (Dia 75):** o extrato, o print das chamadas, o áudio e o papel do "Dr. Silva" viram evidência. Hash em tudo, e uma linha do tempo em UTC.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>